## Model v4 + Frozen Classifier Feature Loss

기존 ECG-pulse correlation loss를 frozen VT classifier feature-matching loss로 교체한 버전.

## 1. Google Drive + vtac 설정

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
VTAC_ZIP = '/content/drive/MyDrive/vtac.zip'
VTAC_OUT = '/content/vtac_unzipped'

if not os.path.exists(VTAC_OUT) or len(os.listdir(VTAC_OUT)) == 0:
    print('vtac.zip 복사 중...')
    t0 = time.time()
    os.makedirs(VTAC_OUT, exist_ok=True)
    os.system(f'cp "{VTAC_ZIP}" /content/vtac.zip')
    os.system(f'unzip -q /content/vtac.zip -d {VTAC_OUT}')
    print(f'완료! ({time.time()-t0:.0f}초)')
else:
    print('이미 압축 해제됨 — 건너뜀')


Mounted at /content/drive
vtac.zip 복사 중...
완료! (122초)


## 2. Import

In [2]:
!pip install PyWavelets scikit-learn -q

import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pywt
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

SEED = 42
def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
print('device:', device)


device: cuda


## 3. 경로 설정

In [3]:
DATA_DIR = '/content/drive/MyDrive/vtac_preprocessed_10s_v2'
TRAIN_PT = f'{DATA_DIR}/train_10s_v2.pt'
VAL_PT   = f'{DATA_DIR}/val_10s_v2.pt'
TEST_PT  = f'{DATA_DIR}/test_10s_v2.pt'

# 기존 baseline 결과와 섞이지 않게 별도 output directory 사용
OUT_DIR = '/content/drive/MyDrive/vtac_project/outputs/cvae_v4_classifier_physloss'
os.makedirs(OUT_DIR, exist_ok=True)

# Classifier2.ipynb에서 학습/저장한 frozen auxiliary classifier checkpoint
AUX_PHYS_CKPT_PATH = '/content/drive/MyDrive/vtac_project/outputs/masked_multimodal_resnet_tcn_classifier/masked_multimodal_resnet_tcn_best.pt'

print('DATA_DIR:', DATA_DIR)
print('OUT_DIR:', OUT_DIR)
print('AUX_PHYS_CKPT_PATH:', AUX_PHYS_CKPT_PATH)
!ls -lh "{DATA_DIR}" 2>/dev/null || echo '❌ Preprocess_v2.ipynb 먼저 실행하세요'
!ls -lh "{AUX_PHYS_CKPT_PATH}" 2>/dev/null || echo '⚠️ Classifier2.ipynb를 먼저 실행하거나 AUX_PHYS_CKPT_PATH를 수정하세요'

DATA_DIR: /content/drive/MyDrive/vtac_preprocessed_10s_v2
OUT_DIR: /content/drive/MyDrive/vtac_project/outputs/cvae_v4_classifier_physloss
AUX_PHYS_CKPT_PATH: /content/drive/MyDrive/vtac_project/outputs/masked_multimodal_resnet_tcn_classifier/masked_multimodal_resnet_tcn_best.pt
total 187M
-rw------- 1 root root  18M Jun  1 17:50 test_10s_v2.pt
-rw------- 1 root root 151M Jun  1 17:50 train_10s_v2.pt
-rw------- 1 root root  19M Jun  1 17:50 val_10s_v2.pt
-rw------- 1 root root 19M Jun 19 11:11 /content/drive/MyDrive/vtac_project/outputs/masked_multimodal_resnet_tcn_classifier/masked_multimodal_resnet_tcn_best.pt


## 4. 데이터 로드

In [4]:
def safe_load(path):
    try:    return torch.load(path, map_location='cpu', weights_only=False)
    except: return torch.load(path, map_location='cpu')

def load_split(path):
    d = safe_load(path)
    X, y, m = d['X'].float(), d['y'].long(), d['m_channel'].float()
    print(path.split('/')[-1], '| X:', tuple(X.shape),
          '| label:', torch.bincount(y, minlength=2).tolist(),
          '| ch avail:', m.sum(0).long().tolist())
    return X, y, m, d

X_train, y_train, m_train, _ = load_split(TRAIN_PT)
X_val,   y_val,   m_val,   _ = load_split(VAL_PT)
X_test,  y_test,  m_test,  _ = load_split(TEST_PT)
CHANNELS = ['ECG1', 'ECG2', 'PPG', 'ABP']


train_10s_v2.pt | X: (3901, 4, 2500) | label: [2754, 1147] | ch avail: [3901, 3901, 3572, 1393]
val_10s_v2.pt | X: (481, 4, 2500) | label: [341, 140] | ch avail: [481, 481, 445, 171]
test_10s_v2.pt | X: (465, 4, 2500) | label: [328, 137] | ch avail: [465, 465, 425, 171]


## 5. Dataset + DataLoader

In [5]:
C_DIM = 5; LATENT_DIM = 128; BATCH_SIZE = 64; SEQ_LEN = 2500

def make_cond(y, m):
    return torch.cat([y.float().unsqueeze(1), m.float()], dim=1)

c_train = make_cond(y_train, m_train)
c_val   = make_cond(y_val,   m_val)
c_test  = make_cond(y_test,  m_test)

train_ds = TensorDataset(X_train, c_train, m_train)
val_ds   = TensorDataset(X_val,   c_val,   m_val)

cw = 1.0 / torch.bincount(y_train, minlength=2).float()
sampler = WeightedRandomSampler(cw[y_train], len(y_train), replacement=True)

train_loader = DataLoader(train_ds, BATCH_SIZE, sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print('train:', len(train_ds), '| val:', len(val_ds))


train: 3901 | val: 481


## 6. 모델 아키텍처
### 6-1. ConvBranchEncoder

In [6]:
class ConvBranchEncoder(nn.Module):
    def __init__(self, in_wave_channels, c_dim, hidden_channels=(32,64,128),
                 out_dim=256, seq_len=2500, norm_groups=8):
        super().__init__()
        self.c_dim   = c_dim
        self.seq_len = seq_len
        ic = in_wave_channels + c_dim

        def gn(ch):
            g = min(norm_groups, ch)
            while ch % g: g -= 1
            return nn.GroupNorm(g, ch)

        self.conv = nn.Sequential(
            nn.Conv1d(ic, hidden_channels[0], 9, stride=2, padding=4), gn(hidden_channels[0]), nn.SiLU(),
            nn.Conv1d(hidden_channels[0], hidden_channels[1], 9, stride=2, padding=4), gn(hidden_channels[1]), nn.SiLU(),
            nn.Conv1d(hidden_channels[1], hidden_channels[2], 7, stride=2, padding=3), gn(hidden_channels[2]), nn.SiLU(),
            nn.Conv1d(hidden_channels[2], hidden_channels[2], 7, stride=1, padding=3), gn(hidden_channels[2]), nn.SiLU(),
        )
        with torch.no_grad():
            d = torch.zeros(1, in_wave_channels + c_dim, seq_len)
            self.flat_dim = self.conv(d).numel()

        self.proj = nn.Sequential(nn.Linear(self.flat_dim, out_dim), nn.LayerNorm(out_dim), nn.SiLU())

    def forward(self, x, c):
        B, _, T = x.shape
        h = torch.cat([x, c.unsqueeze(-1).expand(B, self.c_dim, T)], dim=1)
        return self.proj(self.conv(h).flatten(1))


### 6-2. CrossAttentionFusion

In [7]:
class CrossAttentionFusion(nn.Module):
    def __init__(self, branch_dim=256, c_dim=5, num_heads=8,
                 fusion_dim=512, dropout=0.2):
        super().__init__()
        self.c_proj = nn.Linear(c_dim, branch_dim)
        self.attn   = nn.MultiheadAttention(branch_dim, num_heads,
                                             dropout=dropout, batch_first=True)
        self.norm1  = nn.LayerNorm(branch_dim)
        self.ffn    = nn.Sequential(
            nn.Linear(branch_dim, branch_dim * 4), nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(branch_dim * 4, branch_dim),
        )
        self.norm2    = nn.LayerNorm(branch_dim)
        self.out_proj = nn.Sequential(
            nn.Linear(3 * branch_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.SiLU(),
            nn.Linear(fusion_dim, fusion_dim),     nn.LayerNorm(fusion_dim), nn.SiLU(),
        )

    def forward(self, h_ecg, h_ppg, h_abp, c):
        c_emb  = self.c_proj(c)
        tokens = torch.stack([h_ecg + c_emb, h_ppg + c_emb, h_abp + c_emb], dim=1)
        attn_out, _ = self.attn(tokens, tokens, tokens)
        tokens = self.norm1(tokens + attn_out)
        tokens = self.norm2(tokens + self.ffn(tokens))
        return self.out_proj(tokens.flatten(1))


### 6-3. MultiBranchCVAE

In [8]:
class MultiBranchCVAE(nn.Module):
    def __init__(self, x_channels=4, c_dim=5, latent_dim=128, seq_len=2500,
                 branch_dim=256, fusion_dim=512, num_heads=8, norm_groups=8):
        super().__init__()
        self.x_channels = x_channels
        self.c_dim      = c_dim
        self.latent_dim = latent_dim
        self.seq_len    = seq_len
        self.branch_dim = branch_dim

        self.ecg_encoder = ConvBranchEncoder(2, c_dim, out_dim=branch_dim, seq_len=seq_len, norm_groups=norm_groups)
        self.ppg_encoder = ConvBranchEncoder(1, c_dim, out_dim=branch_dim, seq_len=seq_len, norm_groups=norm_groups)
        self.abp_encoder = ConvBranchEncoder(1, c_dim, out_dim=branch_dim, seq_len=seq_len, norm_groups=norm_groups)

        self.fusion    = CrossAttentionFusion(branch_dim, c_dim, num_heads, fusion_dim)
        self.fc_mu     = nn.Linear(fusion_dim, latent_dim)
        self.fc_logvar = nn.Linear(fusion_dim, latent_dim)

        self.dec_len      = seq_len // 4
        self.dec_channels = 128
        self.fc_dec = nn.Sequential(
            nn.Linear(latent_dim + c_dim, fusion_dim), nn.LayerNorm(fusion_dim), nn.SiLU(),
            nn.Linear(fusion_dim, self.dec_channels * self.dec_len), nn.SiLU(),
        )
        self.decoder = nn.Sequential(
            nn.Conv1d(128, 128, 7, padding=3), nn.SiLU(),
            nn.Upsample(scale_factor=2, mode='linear', align_corners=False),
            nn.Conv1d(128, 64, 7, padding=3), nn.SiLU(),
            nn.Upsample(scale_factor=2, mode='linear', align_corners=False),
            nn.Conv1d(64, 32, 7, padding=3), nn.SiLU(),
            nn.Conv1d(32, x_channels, 9, padding=4),
        )

    def _match_len(self, x, n):
        t = x.shape[-1]
        if t == n: return x
        return x[..., :n] if t > n else F.pad(x, (0, n - t))

    def _gates(self, c):
        if c.size(1) < 5:
            o = torch.ones(c.size(0), 1, device=c.device, dtype=c.dtype)
            return o, o, o
        return torch.clamp(c[:,1:2]+c[:,2:3], 0, 1), c[:,3:4], c[:,4:5]

    def encode(self, x, c):
        g_e, g_p, g_a = self._gates(c)
        h_e = self.ecg_encoder(x[:,0:2,:], c) * g_e
        h_p = self.ppg_encoder(x[:,2:3,:], c) * g_p
        h_a = self.abp_encoder(x[:,3:4,:], c) * g_a
        h   = self.fusion(h_e, h_p, h_a, c)
        mu     = self.fc_mu(h)
        logvar = torch.clamp(self.fc_logvar(h), -10, 10)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * torch.clamp(logvar, -10, 10))

    def decode(self, z, c, m_channel=None):
        h = self.fc_dec(torch.cat([z, c], 1))
        h = h.view(z.size(0), self.dec_channels, self.dec_len)
        x_hat = self._match_len(self.decoder(h), self.seq_len)
        if m_channel is not None:
            x_hat = x_hat * m_channel.unsqueeze(-1)
        return x_hat

    def forward(self, x_enc, c, m_channel=None):
        mu, logvar = self.encode(x_enc, c)
        return self.decode(self.reparameterize(mu, logvar), c, m_channel), mu, logvar

    @torch.no_grad()
    def reconstruct(self, x, c, m_channel=None, use_mean=True):
        self.eval()
        mu, logvar = self.encode(x, c)
        z = mu if use_mean else self.reparameterize(mu, logvar)
        return self.decode(z, c, m_channel)

    @torch.no_grad()
    def sample_prior(self, c, n=None, z_scale=0.7, m_channel=None):
        self.eval()
        if n is None: n = c.size(0)
        if c.size(0) == 1 and n > 1: c = c.expand(n, -1)
        z = torch.randn(n, self.latent_dim, device=c.device, dtype=c.dtype) * z_scale
        return self.decode(z, c, m_channel)


## 6-4. Frozen VT Classifier Feature Encoder

Classifier2.ipynb의 ResNet-TCN classifier를 frozen feature encoder로 불러와 CVAE loss에 사용한다.


In [9]:
class AuxResTCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=7, stride=1, dilation=1, dropout=0.1):
        super().__init__()
        padding = dilation * (kernel_size // 2)

        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size,
                               stride=stride, padding=padding, dilation=dilation, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)

        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=kernel_size,
                               stride=1, padding=padding, dilation=dilation, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.dropout = nn.Dropout(dropout)

        if in_ch != out_ch or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        identity = self.skip(x)

        h = self.conv1(x)
        h = self.bn1(h)
        h = F.silu(h)
        h = self.dropout(h)

        h = self.conv2(h)
        h = self.bn2(h)
        h = h + identity
        h = F.silu(h)
        return h


class AuxModalityEncoder(nn.Module):
    def __init__(self, in_channels, base_ch=64, out_dim=128, dropout=0.1):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_ch, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(base_ch),
            nn.SiLU(),
        )
        self.blocks = nn.Sequential(
            AuxResTCNBlock(base_ch, base_ch, kernel_size=7, stride=1, dilation=1, dropout=dropout),
            AuxResTCNBlock(base_ch, base_ch, kernel_size=7, stride=1, dilation=2, dropout=dropout),
            AuxResTCNBlock(base_ch, base_ch * 2, kernel_size=7, stride=2, dilation=1, dropout=dropout),
            AuxResTCNBlock(base_ch * 2, base_ch * 2, kernel_size=7, stride=1, dilation=2, dropout=dropout),
            AuxResTCNBlock(base_ch * 2, base_ch * 4, kernel_size=7, stride=2, dilation=1, dropout=dropout),
            AuxResTCNBlock(base_ch * 4, base_ch * 4, kernel_size=7, stride=1, dilation=2, dropout=dropout),
            AuxResTCNBlock(base_ch * 4, base_ch * 4, kernel_size=7, stride=1, dilation=4, dropout=dropout),
        )
        self.proj = nn.Sequential(
            nn.Linear(base_ch * 4, out_dim),
            nn.LayerNorm(out_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.stem(x)
        h = self.blocks(h)
        h = h.mean(dim=-1)
        return self.proj(h)


class AuxMaskedMultimodalResNetTCNClassifier(nn.Module):
    """
    Classifier2.ipynb와 동일한 architecture.
    Input:
        x: [B, 4, T] = [ECG1, ECG2, PPG, ABP]
        m: [B, 4]    = channel availability mask
    """
    def __init__(self, branch_dim=128, base_ch=64, dropout=0.15, use_mask_channels=True):
        super().__init__()
        self.branch_dim = branch_dim
        self.use_mask_channels = use_mask_channels

        ecg_in = 4 if use_mask_channels else 2
        ppg_in = 2 if use_mask_channels else 1
        abp_in = 2 if use_mask_channels else 1

        self.ecg_encoder = AuxModalityEncoder(ecg_in, base_ch=base_ch,
                                              out_dim=branch_dim, dropout=dropout)
        self.ppg_encoder = AuxModalityEncoder(ppg_in, base_ch=base_ch // 2,
                                              out_dim=branch_dim, dropout=dropout)
        self.abp_encoder = AuxModalityEncoder(abp_in, base_ch=base_ch // 2,
                                              out_dim=branch_dim, dropout=dropout)

        fusion_in = branch_dim * 3 + 4
        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Dropout(dropout),
        )
        self.head = nn.Linear(128, 1)

    def make_branch_inputs(self, x, m):
        x = x * m.unsqueeze(-1)

        x_ecg = x[:, 0:2, :]
        x_ppg = x[:, 2:3, :]
        x_abp = x[:, 3:4, :]

        if self.use_mask_channels:
            m_ecg_signal = m[:, 0:2].unsqueeze(-1).expand_as(x_ecg)
            m_ppg_signal = m[:, 2:3].unsqueeze(-1).expand_as(x_ppg)
            m_abp_signal = m[:, 3:4].unsqueeze(-1).expand_as(x_abp)

            x_ecg = torch.cat([x_ecg, m_ecg_signal], dim=1)
            x_ppg = torch.cat([x_ppg, m_ppg_signal], dim=1)
            x_abp = torch.cat([x_abp, m_abp_signal], dim=1)

        return x_ecg, x_ppg, x_abp

    def extract_features(self, x, m, return_branches=False):
        x_ecg, x_ppg, x_abp = self.make_branch_inputs(x, m)

        z_ecg = self.ecg_encoder(x_ecg)
        z_ppg = self.ppg_encoder(x_ppg)
        z_abp = self.abp_encoder(x_abp)

        ecg_avail = (m[:, 0] * m[:, 1]).unsqueeze(1)
        ppg_avail = m[:, 2:3]
        abp_avail = m[:, 3:4]

        z_ecg = z_ecg * ecg_avail
        z_ppg = z_ppg * ppg_avail
        z_abp = z_abp * abp_avail

        h = torch.cat([z_ecg, z_ppg, z_abp, m], dim=1)
        feat = self.fusion(h)

        if return_branches:
            return feat, {"z_ecg": z_ecg, "z_ppg": z_ppg, "z_abp": z_abp}
        return feat

    def forward(self, x, m):
        feat = self.extract_features(x, m)
        return self.head(feat).squeeze(1)


def load_frozen_multimodal_resnet_tcn_encoder(ckpt_path, device):
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(
            f"Auxiliary classifier checkpoint not found: {ckpt_path}\n"
            "Classifier2.ipynb를 먼저 실행하거나 AUX_PHYS_CKPT_PATH를 수정하세요."
        )

    try:
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    except TypeError:
        ckpt = torch.load(ckpt_path, map_location=device)

    cfg = ckpt.get("config", {})
    encoder = AuxMaskedMultimodalResNetTCNClassifier(
        branch_dim=cfg.get("branch_dim", 128),
        base_ch=cfg.get("base_ch", 64),
        dropout=cfg.get("dropout", 0.15),
        use_mask_channels=cfg.get("use_mask_channels", True),
    ).to(device)

    encoder.load_state_dict(ckpt["model_state_dict"])
    encoder.eval()

    for p in encoder.parameters():
        p.requires_grad = False

    print("Loaded frozen auxiliary VT classifier")
    print("checkpoint:", ckpt_path)
    print("best val AUPRC:", ckpt.get("best_val_auprc", None))
    print("config:", cfg)
    return encoder


def classifier_phys_loss(phys_encoder, x, x_hat, m, eps=1e-8):
    """
    Frozen classifier feature matching loss.

    x, x_hat: [B, 4, T]
    m       : [B, 4]

    feat_real은 target이므로 no_grad.
    feat_fake는 x_hat 쪽으로 gradient가 흘러야 하므로 no_grad를 걸면 안 됨.
    """
    if phys_encoder is None:
        return torch.tensor(0.0, device=x_hat.device, dtype=x_hat.dtype)

    phys_encoder.eval()

    x = x * m.unsqueeze(-1)
    x_hat = x_hat * m.unsqueeze(-1)

    with torch.no_grad():
        feat_real = phys_encoder.extract_features(x, m)

    feat_fake = phys_encoder.extract_features(x_hat, m)
    denom = feat_real.abs().mean().detach() + eps
    return F.l1_loss(feat_fake, feat_real.detach()) / denom


@torch.no_grad()
def check_phys_encoder(phys_encoder, X_ref, m_ref, device, n=8):
    phys_encoder.eval()
    x = X_ref[:n].to(device).float()
    m = m_ref[:n].to(device).float()
    feat = phys_encoder.extract_features(x, m)
    print("aux check | x:", tuple(x.shape), "m:", tuple(m.shape), "feat:", tuple(feat.shape))
    print("aux feature mean/std:", float(feat.mean()), float(feat.std()))
    return feat


## 7. 손실함수

In [10]:
def masked_recon_loss(x, x_hat, m, eps=1e-8):
    mask = m.to(x.dtype).unsqueeze(-1)
    return ((x - x_hat)**2 * mask).sum() / (mask.sum() * x.shape[-1] + eps)


def masked_derivative_loss(x, x_hat, m, eps=1e-8):
    mask = m.to(x.dtype).unsqueeze(-1)
    dx, dxh = x[:, :, 1:] - x[:, :, :-1], x_hat[:, :, 1:] - x_hat[:, :, :-1]
    return ((dx - dxh)**2 * mask).sum() / (mask.sum() * dx.shape[-1] + eps)


def masked_std_loss(x, x_hat, m, eps=1e-8):
    mask = m.to(x.dtype)
    return ((x.std(-1) - x_hat.std(-1))**2 * mask).sum() / (mask.sum() + eps)


def kl_beta_anneal(epoch, beta_max=1e-3, warmup=30):
    return beta_max * min(1.0, epoch / max(warmup, 1))


def phys_lambda_anneal(epoch, lambda_phys=0.01, warmup=10):
    """Classifier feature loss는 초반 reconstruction이 불안정할 때 과하게 걸리지 않도록 warm-up."""
    return lambda_phys * min(1.0, epoch / max(warmup, 1))


def kl_free_bits_v4(mu, logvar, free_bits=0.1):
    logvar = torch.clamp(logvar, -10.0, 10.0)
    kl_per_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
    threshold  = torch.full_like(kl_per_dim, free_bits).detach()
    return torch.max(kl_per_dim, threshold).mean()


def _haar_dwt1d(x):
    if x.shape[-1] % 2:
        x = F.pad(x, (0, 1))
    return (x[:, :, 0::2] + x[:, :, 1::2]) * 0.5, (x[:, :, 0::2] - x[:, :, 1::2]) * 0.5


def masked_wavelet_loss(x, x_hat, m, n_levels=4, eps=1e-8):
    mask = m.to(x.dtype).unsqueeze(-1)
    total, n = torch.tensor(0., device=x.device, dtype=x.dtype), 0
    cx, cxh = x, x_hat
    for _ in range(n_levels):
        lo, hi   = _haar_dwt1d(cx)
        loh, hih = _haar_dwt1d(cxh)
        total += ((hi - hih).abs() * mask).sum() / (mask.sum() * hi.shape[-1] + eps)
        n += 1
        cx, cxh = lo, loh
    total += ((cx - cxh).abs() * mask).sum() / (mask.sum() * cx.shape[-1] + eps)
    return total / (n + 1)


def cvae_loss_v4(x_clean, x_hat, mu, logvar, m,
                 beta=1e-3, gamma_deriv=0.5, gamma_std=0.1,
                 lambda_wav=0.1, lambda_phys=0.01, free_bits=0.1,
                 phys_encoder=None):
    recon = masked_recon_loss(x_clean, x_hat, m)
    deriv = masked_derivative_loss(x_clean, x_hat, m)
    std   = masked_std_loss(x_clean, x_hat, m)
    wav   = masked_wavelet_loss(x_clean, x_hat, m)

    # 기존 ECG-pulse raw correlation loss 대신 frozen classifier feature matching loss 사용
    phys = classifier_phys_loss(phys_encoder, x_clean, x_hat, m) if lambda_phys > 0 else torch.tensor(0.0, device=x_hat.device, dtype=x_hat.dtype)

    kl    = kl_free_bits_v4(mu, logvar, free_bits)
    loss  = recon + gamma_deriv * deriv + gamma_std * std + lambda_wav * wav + lambda_phys * phys + beta * kl

    return loss, {
        'loss': loss.detach(),
        'recon': recon.detach(),
        'deriv': deriv.detach(),
        'std': std.detach(),
        'wav': wav.detach(),
        'phys': phys.detach(),
        'kl': kl.detach(),
    }

## 8. CVAE 학습 함수 (Denoising Augmentation)

In [11]:
import copy

def augment_signals(x, m_channel, noise_std=0.05):
    noise = torch.randn_like(x) * noise_std
    return x + noise * m_channel.unsqueeze(-1)


def run_epoch_v4(
    model,
    loader,
    optimizer=None,
    device="cuda",
    epoch=1,
    beta_max=1e-3,
    warmup=30,
    gamma_deriv=0.5,
    gamma_std=0.1,
    lambda_wav=0.1,
    lambda_phys=0.01,
    phys_warmup=10,
    grad_clip=1.0,
    free_bits=0.1,
    noise_std=0.05,
    phys_encoder=None,
):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    if phys_encoder is not None:
        phys_encoder.eval()
        for p in phys_encoder.parameters():
            p.requires_grad = False

    beta = kl_beta_anneal(epoch, beta_max, warmup)
    lambda_phys_eff = phys_lambda_anneal(epoch, lambda_phys, phys_warmup)

    totals = {k: 0.0 for k in ["loss", "recon", "deriv", "std", "wav", "phys", "kl"]}
    n_samples = 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for batch in loader:
            x_clean, c, m = [t.to(device).float() for t in batch]
            bs = x_clean.size(0)

            x_enc = (
                augment_signals(x_clean, m, noise_std)
                if (is_train and noise_std > 0)
                else x_clean
            )

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            x_hat, mu, logvar = model(x_enc, c, m_channel=m)

            loss, logs = cvae_loss_v4(
                x_clean,
                x_hat,
                mu,
                logvar,
                m,
                beta=beta,
                gamma_deriv=gamma_deriv,
                gamma_std=gamma_std,
                lambda_wav=lambda_wav,
                lambda_phys=lambda_phys_eff,
                free_bits=free_bits,
                phys_encoder=phys_encoder,
            )

            if is_train:
                loss.backward()

                if grad_clip is not None and grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

                optimizer.step()

            for k in totals:
                totals[k] += logs[k].item() * bs

            n_samples += bs

    r = {k: totals[k] / max(n_samples, 1) for k in totals}
    r["beta"] = beta
    r["lambda_phys_eff"] = lambda_phys_eff

    return r


def train_v4(
    model,
    train_loader,
    val_loader=None,
    device="cuda",
    epochs=60,
    lr=1e-3,
    weight_decay=1e-4,
    beta_max=1e-3,
    warmup=30,
    gamma_deriv=0.5,
    gamma_std=0.1,
    lambda_wav=0.1,
    lambda_phys=0.01,
    phys_warmup=10,
    grad_clip=1.0,
    free_bits=0.1,
    noise_std=0.05,
    save_path=None,
    phys_encoder=None,
):
    model = model.to(device)

    if lambda_phys > 0 and phys_encoder is None:
        raise ValueError("lambda_phys > 0 이면 phys_encoder를 넘겨야 합니다.")

    if phys_encoder is not None:
        phys_encoder = phys_encoder.to(device)
        phys_encoder.eval()
        for p in phys_encoder.parameters():
            p.requires_grad = False

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=epochs,
        eta_min=lr * 0.01,
    )

    history = {
        "train": [],
        "val": [],
    }

    best_val = float("inf")
    best_ep = -1
    best_state_dict = None

    aux_phys_ckpt = globals().get("AUX_PHYS_CKPT_PATH", None)

    for ep in range(1, epochs + 1):
        tr = run_epoch_v4(
            model=model,
            loader=train_loader,
            optimizer=opt,
            device=device,
            epoch=ep,
            beta_max=beta_max,
            warmup=warmup,
            gamma_deriv=gamma_deriv,
            gamma_std=gamma_std,
            lambda_wav=lambda_wav,
            lambda_phys=lambda_phys,
            phys_warmup=phys_warmup,
            grad_clip=grad_clip,
            free_bits=free_bits,
            noise_std=noise_std,
            phys_encoder=phys_encoder,
        )

        sched.step()
        history["train"].append(tr)

        vl = None

        if val_loader is not None:
            vl = run_epoch_v4(
                model=model,
                loader=val_loader,
                optimizer=None,
                device=device,
                epoch=ep,
                beta_max=beta_max,
                warmup=warmup,
                gamma_deriv=gamma_deriv,
                gamma_std=gamma_std,
                lambda_wav=lambda_wav,
                lambda_phys=lambda_phys,
                phys_warmup=phys_warmup,
                grad_clip=grad_clip,
                free_bits=free_bits,
                noise_std=0.0,
                phys_encoder=phys_encoder,
            )

            history["val"].append(vl)

            if vl["loss"] < best_val:
                best_val = vl["loss"]
                best_ep = ep
                best_state_dict = copy.deepcopy(model.state_dict())

                if save_path is not None:
                    torch.save(
                        {
                            "model_state_dict": best_state_dict,
                            "history": history,
                            "epoch": best_ep,
                            "best_val": best_val,
                            "config": {
                                "model": "MultiBranchCVAE_v4_classifier_physloss",
                                "epochs": epochs,
                                "lr": lr,
                                "weight_decay": weight_decay,
                                "beta_max": beta_max,
                                "warmup": warmup,
                                "gamma_deriv": gamma_deriv,
                                "gamma_std": gamma_std,
                                "lambda_wav": lambda_wav,
                                "lambda_phys": lambda_phys,
                                "phys_warmup": phys_warmup,
                                "free_bits": free_bits,
                                "noise_std": noise_std,
                                "aux_phys_ckpt": aux_phys_ckpt,
                            },
                        },
                        save_path,
                    )

        if ep == 1 or ep % 5 == 0:
            msg = (
                f"Ep {ep:03d} | "
                f"tr loss={tr['loss']:.4f} "
                f"recon={tr['recon']:.4f} "
                f"wav={tr['wav']:.4f} "
                f"phys={tr['phys']:.4f} "
                f"lambda_phys={tr['lambda_phys_eff']:.4f} "
                f"kl={tr['kl']:.2f}"
            )

            if vl is not None:
                msg += (
                    f" | val loss={vl['loss']:.4f} "
                    f"recon={vl['recon']:.4f} "
                    f"phys={vl['phys']:.4f}"
                )

            print(msg)

    if val_loader is not None:
        if best_state_dict is None:
            raise RuntimeError("Validation loader가 있는데 best checkpoint가 저장되지 않았습니다.")

        model.load_state_dict(best_state_dict)
        model.eval()

        print(f"Best epoch: {best_ep} | best val loss: {best_val:.6f}")
        print("Returned model has been reloaded with the best validation checkpoint.")

    else:
        best_ep = epochs
        best_val = history["train"][-1]["loss"]

        if save_path is not None:
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "history": history,
                    "epoch": best_ep,
                    "best_train": best_val,
                    "config": {
                        "model": "MultiBranchCVAE_v4_classifier_physloss",
                        "epochs": epochs,
                        "lr": lr,
                        "weight_decay": weight_decay,
                        "beta_max": beta_max,
                        "warmup": warmup,
                        "gamma_deriv": gamma_deriv,
                        "gamma_std": gamma_std,
                        "lambda_wav": lambda_wav,
                        "lambda_phys": lambda_phys,
                        "phys_warmup": phys_warmup,
                        "free_bits": free_bits,
                        "noise_std": noise_std,
                        "aux_phys_ckpt": aux_phys_ckpt,
                    },
                },
                save_path,
            )

        model.eval()
        print(f"No validation loader. Returned final epoch model: epoch {best_ep}")

    return model, history

## 9. 학습 실행

## 9. Ablation Runner

아래 셀은 Proposed CVAE의 architecture / loss / mask ablation을 한 번에 학습하고, 각 variant의 best checkpoint와 generated true-alarm payload를 저장한다.


In [16]:

# ============================================================
# ABLATION RUNNER
# Run after cells 1~8 above have defined:
# - X_train, y_train, m_train / X_val, y_val, m_val
# - ConvBranchEncoder, CrossAttentionFusion
# - cvae_loss_v4, train_v4, load_frozen_multimodal_resnet_tcn_encoder
# ============================================================

import os, json, copy, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

ABL_ROOT = "/content/drive/MyDrive/vtac_project/outputs/cvae_ablation"
os.makedirs(ABL_ROOT, exist_ok=True)

# 빠르게 smoke test 할 때 5~10, 최종 ablation은 60 권장
ABL_EPOCHS = 60
ABL_Z_SCALE = 0.7
FORCE_RETRAIN = False
FORCE_REGENERATE = True

# ------------------------------------------------------------
# 1. Condition / Loader helpers
# ------------------------------------------------------------
def make_cond_ablation(y, m, c_dim=5):
    """
    c_dim=5: [label, ECG1 mask, ECG2 mask, PPG mask, ABP mask]
    c_dim=1: [label] only. Used for w/o channel mask ablation.
    """
    y = y.float().unsqueeze(1)
    if c_dim == 1:
        return y
    if c_dim == 5:
        return torch.cat([y, m.float()], dim=1)
    raise ValueError(f"Unsupported c_dim={c_dim}. Use 1 or 5.")


def build_cvae_loaders(c_dim=5, batch_size=BATCH_SIZE):
    c_tr = make_cond_ablation(y_train, m_train, c_dim=c_dim)
    c_vl = make_cond_ablation(y_val,   m_val,   c_dim=c_dim)

    train_ds_ab = TensorDataset(X_train, c_tr, m_train)
    val_ds_ab   = TensorDataset(X_val,   c_vl, m_val)

    cw_ab = 1.0 / torch.bincount(y_train, minlength=2).float()
    sampler_ab = WeightedRandomSampler(cw_ab[y_train], len(y_train), replacement=True)

    train_loader_ab = DataLoader(
        train_ds_ab, batch_size=batch_size, sampler=sampler_ab,
        num_workers=2, pin_memory=True
    )
    val_loader_ab = DataLoader(
        val_ds_ab, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True
    )
    return train_loader_ab, val_loader_ab


# ------------------------------------------------------------
# 2. Architecture variants
# ------------------------------------------------------------
class ConcatFusion(nn.Module):
    """
    Attention fusion ablation.
    Instead of feature-level cross-attention, simply concatenate modality features and condition.
    """
    def __init__(self, branch_dim=256, c_dim=5, fusion_dim=512, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3 * branch_dim + c_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.SiLU(),
        )

    def forward(self, h_ecg, h_ppg, h_abp, c):
        return self.net(torch.cat([h_ecg, h_ppg, h_abp, c], dim=1))


class MultiBranchCVAEFlexible(nn.Module):
    """
    Multi-branch encoder with selectable fusion:
    - fusion_type="attention": ECG/PPG/ABP tokens interact through cross-attention.
    - fusion_type="concat": modality features are concatenated without attention.
    """
    def __init__(
        self,
        x_channels=4,
        c_dim=5,
        latent_dim=128,
        seq_len=2500,
        branch_dim=256,
        fusion_dim=512,
        num_heads=8,
        norm_groups=8,
        fusion_type="attention",
    ):
        super().__init__()
        self.x_channels = x_channels
        self.c_dim = c_dim
        self.latent_dim = latent_dim
        self.seq_len = seq_len
        self.branch_dim = branch_dim
        self.fusion_type = fusion_type

        self.ecg_encoder = ConvBranchEncoder(2, c_dim, out_dim=branch_dim, seq_len=seq_len, norm_groups=norm_groups)
        self.ppg_encoder = ConvBranchEncoder(1, c_dim, out_dim=branch_dim, seq_len=seq_len, norm_groups=norm_groups)
        self.abp_encoder = ConvBranchEncoder(1, c_dim, out_dim=branch_dim, seq_len=seq_len, norm_groups=norm_groups)

        if fusion_type == "attention":
            self.fusion = CrossAttentionFusion(branch_dim, c_dim, num_heads, fusion_dim)
        elif fusion_type == "concat":
            self.fusion = ConcatFusion(branch_dim, c_dim, fusion_dim)
        else:
            raise ValueError(f"Unknown fusion_type={fusion_type}")

        self.fc_mu = nn.Linear(fusion_dim, latent_dim)
        self.fc_logvar = nn.Linear(fusion_dim, latent_dim)

        self.dec_len = seq_len // 4
        self.dec_channels = 128
        self.fc_dec = nn.Sequential(
            nn.Linear(latent_dim + c_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.SiLU(),
            nn.Linear(fusion_dim, self.dec_channels * self.dec_len),
            nn.SiLU(),
        )
        self.decoder = nn.Sequential(
            nn.Conv1d(128, 128, 7, padding=3),
            nn.SiLU(),
            nn.Upsample(scale_factor=2, mode="linear", align_corners=False),
            nn.Conv1d(128, 64, 7, padding=3),
            nn.SiLU(),
            nn.Upsample(scale_factor=2, mode="linear", align_corners=False),
            nn.Conv1d(64, 32, 7, padding=3),
            nn.SiLU(),
            nn.Conv1d(32, x_channels, 9, padding=4),
        )

    def _match_len(self, x, n):
        t = x.shape[-1]
        if t == n:
            return x
        return x[..., :n] if t > n else F.pad(x, (0, n - t))

    def _gates(self, c):
        # c_dim=1 means label-only condition; branch availability is not given.
        # All branches are treated as potentially available.
        if c.size(1) < 5:
            o = torch.ones(c.size(0), 1, device=c.device, dtype=c.dtype)
            return o, o, o
        return torch.clamp(c[:, 1:2] + c[:, 2:3], 0, 1), c[:, 3:4], c[:, 4:5]

    def encode(self, x, c):
        g_e, g_p, g_a = self._gates(c)
        h_e = self.ecg_encoder(x[:, 0:2, :], c) * g_e
        h_p = self.ppg_encoder(x[:, 2:3, :], c) * g_p
        h_a = self.abp_encoder(x[:, 3:4, :], c) * g_a
        h = self.fusion(h_e, h_p, h_a, c)
        mu = self.fc_mu(h)
        logvar = torch.clamp(self.fc_logvar(h), -10, 10)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * torch.clamp(logvar, -10, 10))

    def decode(self, z, c, m_channel=None):
        h = self.fc_dec(torch.cat([z, c], dim=1))
        h = h.view(z.size(0), self.dec_channels, self.dec_len)
        x_hat = self._match_len(self.decoder(h), self.seq_len)
        if m_channel is not None:
            x_hat = x_hat * m_channel.unsqueeze(-1)
        return x_hat

    def forward(self, x_enc, c, m_channel=None):
        mu, logvar = self.encode(x_enc, c)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, c, m_channel=m_channel), mu, logvar

    @torch.no_grad()
    def reconstruct(self, x, c, m_channel=None, use_mean=True):
        self.eval()
        mu, logvar = self.encode(x, c)
        z = mu if use_mean else self.reparameterize(mu, logvar)
        return self.decode(z, c, m_channel=m_channel)

    @torch.no_grad()
    def sample_prior(self, c, n=None, z_scale=0.7, m_channel=None):
        self.eval()
        if n is None:
            n = c.size(0)
        if c.size(0) == 1 and n > 1:
            c = c.expand(n, -1)
        z = torch.randn(n, self.latent_dim, device=c.device, dtype=c.dtype) * z_scale
        return self.decode(z, c, m_channel=m_channel)


class SingleEncoderCVAE(nn.Module):
    """
    Single encoder ablation.
    All 4 modalities are encoded together instead of using ECG/PPG/ABP-specific branches.
    """
    def __init__(
        self,
        x_channels=4,
        c_dim=5,
        latent_dim=128,
        seq_len=2500,
        branch_dim=256,
        fusion_dim=512,
        norm_groups=8,
    ):
        super().__init__()
        self.x_channels = x_channels
        self.c_dim = c_dim
        self.latent_dim = latent_dim
        self.seq_len = seq_len

        self.encoder = ConvBranchEncoder(
            in_wave_channels=x_channels,
            c_dim=c_dim,
            hidden_channels=(32, 64, 128),
            out_dim=fusion_dim,
            seq_len=seq_len,
            norm_groups=norm_groups,
        )
        self.fc_mu = nn.Linear(fusion_dim, latent_dim)
        self.fc_logvar = nn.Linear(fusion_dim, latent_dim)

        self.dec_len = seq_len // 4
        self.dec_channels = 128
        self.fc_dec = nn.Sequential(
            nn.Linear(latent_dim + c_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.SiLU(),
            nn.Linear(fusion_dim, self.dec_channels * self.dec_len),
            nn.SiLU(),
        )
        self.decoder = nn.Sequential(
            nn.Conv1d(128, 128, 7, padding=3),
            nn.SiLU(),
            nn.Upsample(scale_factor=2, mode="linear", align_corners=False),
            nn.Conv1d(128, 64, 7, padding=3),
            nn.SiLU(),
            nn.Upsample(scale_factor=2, mode="linear", align_corners=False),
            nn.Conv1d(64, 32, 7, padding=3),
            nn.SiLU(),
            nn.Conv1d(32, x_channels, 9, padding=4),
        )

    def _match_len(self, x, n):
        t = x.shape[-1]
        if t == n:
            return x
        return x[..., :n] if t > n else F.pad(x, (0, n - t))

    def encode(self, x, c):
        h = self.encoder(x, c)
        mu = self.fc_mu(h)
        logvar = torch.clamp(self.fc_logvar(h), -10, 10)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * torch.clamp(logvar, -10, 10))

    def decode(self, z, c, m_channel=None):
        h = self.fc_dec(torch.cat([z, c], dim=1))
        h = h.view(z.size(0), self.dec_channels, self.dec_len)
        x_hat = self._match_len(self.decoder(h), self.seq_len)
        if m_channel is not None:
            x_hat = x_hat * m_channel.unsqueeze(-1)
        return x_hat

    def forward(self, x_enc, c, m_channel=None):
        mu, logvar = self.encode(x_enc, c)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, c, m_channel=m_channel), mu, logvar

    @torch.no_grad()
    def reconstruct(self, x, c, m_channel=None, use_mean=True):
        self.eval()
        mu, logvar = self.encode(x, c)
        z = mu if use_mean else self.reparameterize(mu, logvar)
        return self.decode(z, c, m_channel=m_channel)

    @torch.no_grad()
    def sample_prior(self, c, n=None, z_scale=0.7, m_channel=None):
        self.eval()
        if n is None:
            n = c.size(0)
        if c.size(0) == 1 and n > 1:
            c = c.expand(n, -1)
        z = torch.randn(n, self.latent_dim, device=c.device, dtype=c.dtype) * z_scale
        return self.decode(z, c, m_channel=m_channel)


def build_ablation_model(cfg):
    c_dim = cfg.get("c_dim", 5)
    arch = cfg["arch"]

    if arch == "single":
        return SingleEncoderCVAE(
            x_channels=4,
            c_dim=c_dim,
            latent_dim=LATENT_DIM,
            seq_len=SEQ_LEN,
            branch_dim=cfg.get("branch_dim", 256),
            fusion_dim=cfg.get("fusion_dim", 512),
        )

    if arch == "multi_attention":
        return MultiBranchCVAEFlexible(
            x_channels=4,
            c_dim=c_dim,
            latent_dim=LATENT_DIM,
            seq_len=SEQ_LEN,
            branch_dim=cfg.get("branch_dim", 256),
            fusion_dim=cfg.get("fusion_dim", 512),
            num_heads=cfg.get("num_heads", 8),
            fusion_type="attention",
        )

    if arch == "multi_concat":
        return MultiBranchCVAEFlexible(
            x_channels=4,
            c_dim=c_dim,
            latent_dim=LATENT_DIM,
            seq_len=SEQ_LEN,
            branch_dim=cfg.get("branch_dim", 256),
            fusion_dim=cfg.get("fusion_dim", 512),
            num_heads=cfg.get("num_heads", 8),
            fusion_type="concat",
        )

    raise ValueError(f"Unknown arch={arch}")


# ------------------------------------------------------------
# 3. Generation / diagnostics
# ------------------------------------------------------------
@torch.no_grad()
def generate_ablation(
    model,
    true_masks,
    n,
    c_dim=5,
    z_scale=0.7,
    bs=128,
    device="cuda",
    seed=42,
    use_output_mask_in_decode=True,
):
    model.eval()
    if isinstance(true_masks, np.ndarray):
        true_masks = torch.tensor(true_masks, dtype=torch.float32)

    rng = np.random.default_rng(seed)
    idx = rng.choice(len(true_masks), size=n, replace=True)
    masks = true_masks[idx].float()

    if c_dim == 1:
        cond = torch.ones(n, 1)
    elif c_dim == 5:
        cond = torch.cat([torch.ones(n, 1), masks], dim=1)
    else:
        raise ValueError(f"Unsupported c_dim={c_dim}")

    outs = []
    for s in range(0, n, bs):
        cb = cond[s:s+bs].to(device)
        mb = masks[s:s+bs].to(device)

        # For w/o mask condition, set use_output_mask_in_decode=False to avoid leaking mask to decoder.
        dec_mask = mb if use_output_mask_in_decode else None
        xb = model.sample_prior(cb, n=cb.size(0), z_scale=z_scale, m_channel=dec_mask)

        # Final saved payload is still masked because missing channels are not observed.
        xb = xb * mb.unsqueeze(-1)
        outs.append(xb.detach().cpu())

    return torch.cat(outs, dim=0), masks


@torch.no_grad()
def latent_diag_dict(model, X, y, m, c_dim=5, n=512, device="cuda"):
    model.eval()
    true_idx = torch.where(y == 1)[0]
    if len(true_idx) == 0:
        return {}

    perm = torch.randperm(len(true_idx))[:min(n, len(true_idx))]
    idx = true_idx[perm]

    xb = X[idx].to(device).float()
    cb = make_cond_ablation(y[idx], m[idx], c_dim=c_dim).to(device).float()

    mu, logvar = model.encode(xb, cb)
    std = torch.exp(0.5 * torch.clamp(logvar, -10, 10))

    return {
        "mu_mean": float(mu.mean().detach().cpu()),
        "mu_std": float(mu.std().detach().cpu()),
        "std_mean": float(std.mean().detach().cpu()),
        "std_std": float(std.std().detach().cpu()),
        "logvar_mean": float(logvar.mean().detach().cpu()),
    }


def sample_stats_dict(X_syn, m_syn):
    stats = {
        "n_syn": int(X_syn.size(0)),
        "global_mean": float(X_syn.mean()),
        "global_std": float(X_syn.std()),
        "global_min": float(X_syn.min()),
        "global_max": float(X_syn.max()),
    }
    ch_names = CHANNELS if "CHANNELS" in globals() else ["ECG1", "ECG2", "PPG", "ABP"]

    for ci, ch in enumerate(ch_names):
        avail = m_syn[:, ci] > 0.5
        stats[f"{ch}_avail_n"] = int(avail.sum())
        if int(avail.sum()) > 0:
            vals = X_syn[avail, ci, :]
            stats[f"{ch}_std"] = float(vals.std())
            stats[f"{ch}_mean"] = float(vals.mean())
        else:
            stats[f"{ch}_std"] = None
            stats[f"{ch}_mean"] = None
    return stats


# ------------------------------------------------------------
# 4. Ablation configs
# ------------------------------------------------------------
# current baseline/proposed setting:
BASE_LOSS_KW = dict(
    gamma_deriv=0.5,
    gamma_std=0.1,
    lambda_wav=0.1,
    lambda_phys=0.01,   # In this notebook: frozen VT classifier feature loss, not raw ECG-pulse corr.
    beta_max=1e-3,
    warmup=30,
    phys_warmup=10,
    grad_clip=1.0,
    free_bits=0.1,
    noise_std=0.05,
)

ABLATIONS = [
    # Reference full proposed. Keep this if you want a fully matched reference run.
    dict(
        name="full_proposed",
        question="Reference: multi-branch + attention + all losses + channel mask",
        arch="multi_attention",
        c_dim=5,
        use_output_mask_in_decode=True,
        **BASE_LOSS_KW,
    ),

    # Architecture ablation 1
    dict(
        name="arch_single_encoder",
        question="Single encoder vs Multi-branch encoder",
        arch="single",
        c_dim=5,
        use_output_mask_in_decode=True,
        **BASE_LOSS_KW,
    ),

    # Architecture ablation 2
    dict(
        name="arch_concat_fusion",
        question="Concatenation fusion vs Attention fusion",
        arch="multi_concat",
        c_dim=5,
        use_output_mask_in_decode=True,
        **BASE_LOSS_KW,
    ),

    # Loss ablation 1: coefficient-domain / wavelet loss
    dict(
        name="loss_no_coeff_wavelet",
        question="w/o coefficient-domain loss",
        arch="multi_attention",
        c_dim=5,
        use_output_mask_in_decode=True,
        **{**BASE_LOSS_KW, "lambda_wav": 0.0},
    ),

    # Loss ablation 2: derivative / slope loss
    dict(
        name="loss_no_derivative",
        question="w/o derivative loss",
        arch="multi_attention",
        c_dim=5,
        use_output_mask_in_decode=True,
        **{**BASE_LOSS_KW, "gamma_deriv": 0.0},
    ),

    # Loss ablation 3: current phys term = frozen classifier feature matching
    # If you later restore a raw ECG-pulse coupling loss, replace cvae_loss_v4 accordingly.
    dict(
        name="loss_no_phys_feature",
        question="w/o ECG-pulse / classifier feature consistency loss",
        arch="multi_attention",
        c_dim=5,
        use_output_mask_in_decode=True,
        **{**BASE_LOSS_KW, "lambda_phys": 0.0},
    ),

    # Mask ablation
    dict(
        name="mask_no_channel_mask_condition",
        question="w/o channel mask condition",
        arch="multi_attention",
        c_dim=1,
        use_output_mask_in_decode=False,  # avoid giving m to decoder; final saved signals are still masked.
        **BASE_LOSS_KW,
    ),
]

# 원하는 것만 먼저 돌릴 때 여기 수정
ABL_EPOCHS = 60
FORCE_RETRAIN = False
FORCE_REGENERATE = True

RUN_ABLATIONS = [
    "full_proposed",
]

ABLATIONS_TO_RUN = [cfg for cfg in ABLATIONS if cfg["name"] in RUN_ABLATIONS]
print("Ablations to run:")
for cfg in ABLATIONS_TO_RUN:
    print("-", cfg["name"], "|", cfg["question"])


# ------------------------------------------------------------
# 5. Main runner
# ------------------------------------------------------------
def run_one_ablation(cfg):
    set_seed(SEED)

    name = cfg["name"]
    out_dir = os.path.join(ABL_ROOT, name)
    os.makedirs(out_dir, exist_ok=True)

    ckpt_path = os.path.join(out_dir, f"{name}_best.pt")
    syn_path  = os.path.join(out_dir, f"generated_true_{name}.pt")
    meta_path = os.path.join(out_dir, f"{name}_meta.json")

    print("\n" + "=" * 80)
    print(f"[ABLATION] {name}")
    print(cfg["question"])
    print("out_dir:", out_dir)
    print("=" * 80)

    train_loader_ab, val_loader_ab = build_cvae_loaders(c_dim=cfg.get("c_dim", 5))
    model_ab = build_ablation_model(cfg).to(device)

    n_params = sum(p.numel() for p in model_ab.parameters() if p.requires_grad)
    print(f"Trainable params: {n_params:,}")

    # phys_encoder only required when lambda_phys > 0
    phys_encoder_use = None
    if cfg.get("lambda_phys", 0.0) > 0:
        if "phys_encoder_ablation" not in globals():
            raise RuntimeError("phys_encoder_ablation이 없음. 아래 main block에서 먼저 로드하세요.")
        phys_encoder_use = phys_encoder_ablation

    if (not os.path.exists(ckpt_path)) or FORCE_RETRAIN:
        start = time.time()
        model_ab, hist_ab = train_v4(
            model=model_ab,
            train_loader=train_loader_ab,
            val_loader=val_loader_ab,
            device=device,
            epochs=ABL_EPOCHS,
            lr=1e-3,
            weight_decay=1e-4,
            beta_max=cfg["beta_max"],
            warmup=cfg["warmup"],
            gamma_deriv=cfg["gamma_deriv"],
            gamma_std=cfg["gamma_std"],
            lambda_wav=cfg["lambda_wav"],
            lambda_phys=cfg["lambda_phys"],
            phys_warmup=cfg["phys_warmup"],
            grad_clip=cfg["grad_clip"],
            free_bits=cfg["free_bits"],
            noise_std=cfg["noise_std"],
            save_path=ckpt_path,
            phys_encoder=phys_encoder_use,
        )
        train_sec = time.time() - start
    else:
        print("Checkpoint exists. Loading:", ckpt_path)
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model_ab.load_state_dict(ckpt["model_state_dict"])
        model_ab.eval()
        hist_ab = ckpt.get("history", None)
        train_sec = None

    # Re-save checkpoint with ablation config metadata
    try:
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        ckpt["ablation_config"] = cfg
        ckpt["ablation_name"] = name
        ckpt["trainable_params"] = n_params
        torch.save(ckpt, ckpt_path)
        best_epoch = ckpt.get("epoch", None)
        best_val = ckpt.get("best_val", None)
    except Exception as e:
        print("checkpoint metadata update failed:", repr(e))
        best_epoch, best_val = None, None

    # Generate fixed synthetic true samples
    y_train_cpu = y_train.detach().cpu().long()
    m_train_cpu = m_train.detach().cpu().float()
    n_false = int((y_train_cpu == 0).sum())
    n_true = int((y_train_cpu == 1).sum())
    n_gen = n_false - n_true
    true_mask_pool = m_train_cpu[y_train_cpu == 1]

    if (not os.path.exists(syn_path)) or FORCE_REGENERATE:
        X_syn_ab, m_syn_ab = generate_ablation(
            model=model_ab,
            true_masks=true_mask_pool,
            n=n_gen,
            c_dim=cfg.get("c_dim", 5),
            z_scale=ABL_Z_SCALE,
            bs=128,
            device=device,
            seed=SEED,
            use_output_mask_in_decode=cfg.get("use_output_mask_in_decode", True),
        )
        y_syn_ab = torch.ones(X_syn_ab.size(0), dtype=torch.long)

        payload = {
            "X_syn": X_syn_ab.float(),
            "y_syn": y_syn_ab.long(),
            "m_syn": m_syn_ab.float(),
            "method": name,
            "question": cfg["question"],
            "source_ckpt": ckpt_path,
            "best_epoch": best_epoch,
            "best_val": float(best_val) if best_val is not None else None,
            "ablation_config": cfg,
            "channels": CHANNELS if "CHANNELS" in globals() else ["ECG1", "ECG2", "PPG", "ABP"],
            "fs": 250,
            "window_sec": [-10, 0],
            "z_scale": ABL_Z_SCALE,
            "n_false_train": n_false,
            "n_true_train": n_true,
            "n_generated": n_gen,
        }
        torch.save(payload, syn_path)
        print("Saved synthetic:", syn_path)
    else:
        payload = torch.load(syn_path, map_location="cpu", weights_only=False)
        X_syn_ab, m_syn_ab = payload["X_syn"], payload["m_syn"]
        print("Synthetic exists:", syn_path)

    latent_stats = latent_diag_dict(
        model_ab,
        X_train,
        y_train,
        m_train,
        c_dim=cfg.get("c_dim", 5),
        n=512,
        device=device,
    )
    sample_stats = sample_stats_dict(X_syn_ab, m_syn_ab)

    meta = {
        "name": name,
        "question": cfg["question"],
        "ckpt_path": ckpt_path,
        "synthetic_path": syn_path,
        "best_epoch": best_epoch,
        "best_val": float(best_val) if best_val is not None else None,
        "train_sec": train_sec,
        "trainable_params": n_params,
        "latent_stats": latent_stats,
        "sample_stats": sample_stats,
        "config": cfg,
    }

    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print("[latent]", latent_stats)
    print("[sample std]", {k: sample_stats[k] for k in sample_stats if k.endswith("_std") or k == "global_std"})
    print("Saved meta:", meta_path)

    # cleanup
    del model_ab
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return meta


# Load frozen auxiliary classifier once, only if any selected ablation uses lambda_phys > 0
need_phys = any(cfg.get("lambda_phys", 0.0) > 0 for cfg in ABLATIONS_TO_RUN)
if need_phys:
    print("\nLoading frozen auxiliary classifier for feature loss...")
    phys_encoder_ablation = load_frozen_multimodal_resnet_tcn_encoder(AUX_PHYS_CKPT_PATH, device)
    _ = check_phys_encoder(phys_encoder_ablation, X_train, m_train, device, n=8)
else:
    phys_encoder_ablation = None

ablation_results = []
for cfg in ABLATIONS_TO_RUN:
    meta = run_one_ablation(cfg)
    ablation_results.append(meta)

# Save global manifest
manifest = {
    "root": ABL_ROOT,
    "epochs": ABL_EPOCHS,
    "z_scale": ABL_Z_SCALE,
    "seed": SEED,
    "results": ablation_results,
}
manifest_path = os.path.join(ABL_ROOT, "ablation_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("\nDONE.")
print("Manifest:", manifest_path)

# compact table
try:
    import pandas as pd
    rows = []
    for r in ablation_results:
        row = {
            "name": r["name"],
            "best_val": r["best_val"],
            "best_epoch": r["best_epoch"],
            "mu_std": r["latent_stats"].get("mu_std"),
            "std_mean": r["latent_stats"].get("std_mean"),
            "global_std": r["sample_stats"].get("global_std"),
            "ECG1_std": r["sample_stats"].get("ECG1_std"),
            "ECG2_std": r["sample_stats"].get("ECG2_std"),
            "PPG_std": r["sample_stats"].get("PPG_std"),
            "ABP_std": r["sample_stats"].get("ABP_std"),
            "synthetic_path": r["synthetic_path"],
        }
        rows.append(row)
    ablation_df = pd.DataFrame(rows)
    display(ablation_df)
    ablation_df.to_csv(os.path.join(ABL_ROOT, "ablation_summary.csv"), index=False)
except Exception as e:
    print("summary table failed:", repr(e))


Ablations to run:
- full_proposed | Reference: multi-branch + attention + all losses + channel mask

Loading frozen auxiliary classifier for feature loss...
Loaded frozen auxiliary VT classifier
checkpoint: /content/drive/MyDrive/vtac_project/outputs/masked_multimodal_resnet_tcn_classifier/masked_multimodal_resnet_tcn_best.pt
best val AUPRC: 0.9145302423293902
config: {'model': 'MaskedMultimodalResNetTCNClassifier', 'branch_dim': 128, 'base_ch': 64, 'dropout': 0.15, 'use_mask_channels': True, 'pos_weight': 2.4010462074978203, 'channels': ['ECG1', 'ECG2', 'PPG', 'ABP']}
aux check | x: (8, 4, 2500) m: (8, 4) feat: (8, 128)
aux feature mean/std: 0.215154767036438 0.6268650889396667

[ABLATION] full_proposed
Reference: multi-branch + attention + all losses + channel mask
out_dir: /content/drive/MyDrive/vtac_project/outputs/cvae_ablation/full_proposed
Trainable params: 74,229,924
Ep 001 | tr loss=1.0318 recon=0.9328 wav=0.2112 phys=1.1784 lambda_phys=0.0010 kl=0.62 | val loss=0.9806 recon=0

,name,best_val,best_epoch,mu_std,std_mean,global_std,ECG1_std,ECG2_std,PPG_std,ABP_std,synthetic_path
0,full_proposed,0.844836,5,1.017628,0.050528,0.446918,0.399416,0.429522,0.592165,0.621812,/content/drive/MyDrive/vtac_project/outputs/cv...


In [ ]:
# ============================================================
# 10. Ablation Post-hoc Analysis Block
# ============================================================
# Purpose:
# 1) Generator training metric
# 2) Latent diagnostic
# 3) Generated sample statistics
# 4) Realism metrics: waveform plot / PSD distance / MMD / lag consistency / PCA
# 5) Utility manifest for downstream CNN1D/FCN evaluation
# ============================================================

import os, json, glob, math, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
ABL_ROOT = globals().get(
    "ABL_ROOT",
    "/content/drive/MyDrive/vtac_project/outputs/cvae_ablation"
)

ANALYSIS_ROOT = os.path.join(ABL_ROOT, "posthoc_analysis")
os.makedirs(ANALYSIS_ROOT, exist_ok=True)

PLOT_ROOT = os.path.join(ANALYSIS_ROOT, "plots")
os.makedirs(PLOT_ROOT, exist_ok=True)

CHANNELS = globals().get("CHANNELS", ["ECG1", "ECG2", "PPG", "ABP"])
FS = 250
MAX_REALISM_N = 400
MMD_N = 300
PLOT_N = 3

device = globals().get("device", "cuda" if torch.cuda.is_available() else "cpu")

print("ABL_ROOT:", ABL_ROOT)
print("ANALYSIS_ROOT:", ANALYSIS_ROOT)
print("device:", device)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def load_all_ablation_meta(root):
    meta_paths = sorted(glob.glob(os.path.join(root, "*", "*_meta.json")))
    rows = []
    for mp in meta_paths:
        with open(mp, "r") as f:
            meta = json.load(f)
        rows.append(meta)
    if len(rows) == 0:
        raise FileNotFoundError(f"No *_meta.json found under {root}")
    return rows


def extract_best_val_components(ckpt_path):
    """
    Extract best validation component losses from checkpoint history.
    train_v4 saves checkpoint whenever val loss improves, so history['val'][-1]
    is usually the best epoch record.
    """
    if not os.path.exists(ckpt_path):
        return {}

    ckpt = safe_torch_load(ckpt_path, map_location="cpu")
    hist = ckpt.get("history", {})
    val_hist = hist.get("val", [])

    if len(val_hist) == 0:
        return {}

    best_val_log = val_hist[-1]
    return {
        "val_loss": best_val_log.get("loss", np.nan),
        "val_recon": best_val_log.get("recon", np.nan),
        "val_deriv": best_val_log.get("deriv", np.nan),
        "val_std": best_val_log.get("std", np.nan),
        "val_wav": best_val_log.get("wav", np.nan),
        "val_phys": best_val_log.get("phys", np.nan),
        "val_kl": best_val_log.get("kl", np.nan),
        "val_beta": best_val_log.get("beta", np.nan),
        "val_lambda_phys_eff": best_val_log.get("lambda_phys_eff", np.nan),
    }


def get_real_true_reference():
    """
    Prefer held-out test true samples if available.
    Fall back to validation true samples.
    """
    if all(k in globals() for k in ["X_test", "y_test", "m_test"]):
        Xr, yr, mr = X_test, y_test, m_test
        split_name = "test_true"
    elif all(k in globals() for k in ["X_val", "y_val", "m_val"]):
        Xr, yr, mr = X_val, y_val, m_val
        split_name = "val_true"
    else:
        Xr, yr, mr = X_train, y_train, m_train
        split_name = "train_true"

    idx = torch.where(yr.long() == 1)[0]
    Xr = Xr[idx].detach().cpu().float()
    mr = mr[idx].detach().cpu().float()

    if Xr.size(0) > MAX_REALISM_N:
        g = torch.Generator().manual_seed(42)
        perm = torch.randperm(Xr.size(0), generator=g)[:MAX_REALISM_N]
        Xr = Xr[perm]
        mr = mr[perm]

    return Xr, mr, split_name


def load_synthetic_payload(path):
    payload = safe_torch_load(path, map_location="cpu")
    Xs = payload["X_syn"].detach().cpu().float()
    ms = payload["m_syn"].detach().cpu().float()

    if Xs.size(0) > MAX_REALISM_N:
        g = torch.Generator().manual_seed(42)
        perm = torch.randperm(Xs.size(0), generator=g)[:MAX_REALISM_N]
        Xs = Xs[perm]
        ms = ms[perm]

    return Xs, ms, payload


def masked_channel_values(X, m, ch_idx):
    avail = m[:, ch_idx] > 0.5
    if int(avail.sum()) == 0:
        return None
    return X[avail, ch_idx, :]


def compute_psd_distance(real_X, real_m, syn_X, syn_m, eps=1e-8):
    """
    Mean absolute log-PSD distance per channel.
    Lower is better.
    """
    out = {}
    distances = []

    for ci, ch in enumerate(CHANNELS):
        xr = masked_channel_values(real_X, real_m, ci)
        xs = masked_channel_values(syn_X, syn_m, ci)

        if xr is None or xs is None:
            out[f"psd_{ch}"] = np.nan
            continue

        xr_np = xr.numpy()
        xs_np = xs.numpy()

        pr = np.abs(np.fft.rfft(xr_np, axis=-1)) ** 2
        ps = np.abs(np.fft.rfft(xs_np, axis=-1)) ** 2

        pr = np.log(pr.mean(axis=0) + eps)
        ps = np.log(ps.mean(axis=0) + eps)

        d = float(np.mean(np.abs(pr - ps)))
        out[f"psd_{ch}"] = d
        distances.append(d)

    out["psd_mean"] = float(np.nanmean(distances)) if len(distances) else np.nan
    return out


def extract_feature_matrix(X, m, max_n=None):
    """
    Compact statistical + spectral feature vector for MMD/PCA.
    Shape: [N, feature_dim]
    """
    X = X.detach().cpu().float()
    m = m.detach().cpu().float()

    if max_n is not None and X.size(0) > max_n:
        g = torch.Generator().manual_seed(123)
        idx = torch.randperm(X.size(0), generator=g)[:max_n]
        X = X[idx]
        m = m[idx]

    feats = []

    for i in range(X.size(0)):
        xi = X[i]
        mi = m[i]
        f = []

        # channel availability
        f.extend(mi.numpy().tolist())

        for ci in range(X.size(1)):
            xci = xi[ci].numpy()
            if mi[ci] <= 0.5:
                f.extend([0.0] * 9)
                continue

            dx = np.diff(xci)
            spec = np.abs(np.fft.rfft(xci)) ** 2
            spec = spec / (spec.sum() + 1e-8)

            # simple spectral bands
            n_freq = len(spec)
            b1 = spec[: max(1, n_freq // 20)].sum()
            b2 = spec[n_freq // 20 : max(n_freq // 5, n_freq // 20 + 1)].sum()
            b3 = spec[n_freq // 5 :].sum()

            f.extend([
                float(np.mean(xci)),
                float(np.std(xci)),
                float(np.min(xci)),
                float(np.max(xci)),
                float(np.max(xci) - np.min(xci)),
                float(np.std(dx)),
                float(np.mean(np.abs(dx))),
                float(b1),
                float(b2 + b3),
            ])

        feats.append(f)

    return np.asarray(feats, dtype=np.float32)


def rbf_mmd2(X, Y, max_n=300, eps=1e-8):
    """
    RBF-MMD^2 with median heuristic.
    Lower is better.
    """
    if X.shape[0] > max_n:
        X = X[np.random.default_rng(42).choice(X.shape[0], max_n, replace=False)]
    if Y.shape[0] > max_n:
        Y = Y[np.random.default_rng(43).choice(Y.shape[0], max_n, replace=False)]

    Z = np.vstack([X, Y])
    # median heuristic on pairwise distances
    diffs = Z[:, None, :] - Z[None, :, :]
    D2 = np.sum(diffs * diffs, axis=-1)
    med = np.median(D2[D2 > 0])
    sigma2 = med if np.isfinite(med) and med > eps else 1.0

    def kernel(A, B):
        D2 = np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=-1)
        return np.exp(-D2 / (2.0 * sigma2 + eps))

    Kxx = kernel(X, X)
    Kyy = kernel(Y, Y)
    Kxy = kernel(X, Y)

    m = X.shape[0]
    n = Y.shape[0]

    # biased estimator; stable for reporting comparison
    return float(Kxx.mean() + Kyy.mean() - 2.0 * Kxy.mean())


def norm_signal(x, eps=1e-8):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.mean()) / (x.std() + eps)


def best_positive_lag_corr(ecg, pulse, fs=250, max_lag_sec=0.8):
    """
    Positive lag means pulse follows ECG.
    Returns peak corr and lag_sec.
    """
    ecg = norm_signal(ecg)
    pulse = norm_signal(pulse)
    max_lag = int(max_lag_sec * fs)

    best_corr = -np.inf
    best_lag = 0

    for lag in range(0, max_lag + 1):
        if lag == 0:
            a, b = ecg, pulse
        else:
            a, b = ecg[:-lag], pulse[lag:]

        if len(a) < 10:
            continue

        corr = float(np.mean(a * b))
        if corr > best_corr:
            best_corr = corr
            best_lag = lag

    return best_corr, best_lag / fs


def ecg_reference_signal(xi, mi):
    """
    Average available ECG1/ECG2.
    """
    ecgs = []
    if mi[0] > 0.5:
        ecgs.append(xi[0].numpy())
    if mi[1] > 0.5:
        ecgs.append(xi[1].numpy())

    if len(ecgs) == 0:
        return None
    return np.mean(np.stack(ecgs, axis=0), axis=0)


def compute_lag_consistency(X, m, fs=250, max_n=300):
    """
    ECG-to-PPG/ABP lag consistency.
    Returns median peak corr, median lag, and physiological-lag rate.
    """
    X = X.detach().cpu().float()
    m = m.detach().cpu().float()

    if X.size(0) > max_n:
        g = torch.Generator().manual_seed(777)
        idx = torch.randperm(X.size(0), generator=g)[:max_n]
        X = X[idx]
        m = m[idx]

    results = {}

    for target_idx, target_name in [(2, "PPG"), (3, "ABP")]:
        corrs = []
        lags = []

        for i in range(X.size(0)):
            xi, mi = X[i], m[i]
            if mi[target_idx] <= 0.5:
                continue

            ecg = ecg_reference_signal(xi, mi)
            if ecg is None:
                continue

            pulse = xi[target_idx].numpy()
            corr, lag = best_positive_lag_corr(ecg, pulse, fs=fs)
            if np.isfinite(corr):
                corrs.append(corr)
                lags.append(lag)

        if len(corrs) == 0:
            results[f"{target_name}_peak_corr_median"] = np.nan
            results[f"{target_name}_lag_median_sec"] = np.nan
            results[f"{target_name}_phys_lag_rate"] = np.nan
            continue

        corrs = np.asarray(corrs)
        lags = np.asarray(lags)

        # broad physiological lag window; adjust if needed
        phys = (lags >= 0.08) & (lags <= 0.60)

        results[f"{target_name}_peak_corr_median"] = float(np.median(corrs))
        results[f"{target_name}_lag_median_sec"] = float(np.median(lags))
        results[f"{target_name}_phys_lag_rate"] = float(np.mean(phys))

    return results


def plot_waveform_overlay(real_X, real_m, syn_X, syn_m, name, n=PLOT_N):
    """
    Save real-vs-generated waveform overlays.
    """
    out_dir = os.path.join(PLOT_ROOT, "waveform_overlay")
    os.makedirs(out_dir, exist_ok=True)

    n = min(n, real_X.size(0), syn_X.size(0))
    t = np.arange(real_X.size(-1)) / FS

    for k in range(n):
        fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True)

        for ci, ch in enumerate(CHANNELS):
            ax = axes[ci]

            if real_m[k, ci] > 0.5:
                ax.plot(t, real_X[k, ci].numpy(), label="Real", linewidth=1.2)

            if syn_m[k, ci] > 0.5:
                ax.plot(t, syn_X[k, ci].numpy(), label="Generated", linewidth=1.0, alpha=0.8)

            ax.set_title(ch, loc="left", fontsize=9)
            ax.grid(alpha=0.2)

        axes[0].legend(loc="upper right")
        axes[-1].set_xlabel("Time (sec)")
        fig.suptitle(f"Real vs Generated Overlay: {name} | sample {k}")
        fig.tight_layout()

        path = os.path.join(out_dir, f"{name}_overlay_{k}.png")
        fig.savefig(path, dpi=160)
        plt.close(fig)


def plot_reconstruction_overlay_for_variant(meta, n=PLOT_N):
    """
    Real vs reconstructed overlay from best checkpoint.
    Requires build_ablation_model and make_cond_ablation from previous cells.
    """
    if "build_ablation_model" not in globals():
        return None

    ckpt_path = meta["ckpt_path"]
    cfg = meta["config"]

    if not os.path.exists(ckpt_path):
        return None

    out_dir = os.path.join(PLOT_ROOT, "reconstruction_overlay")
    os.makedirs(out_dir, exist_ok=True)

    Xr, mr, split_name = get_real_true_reference()
    n = min(n, Xr.size(0))

    model = build_ablation_model(cfg).to(device)
    ckpt = safe_torch_load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    yr = torch.ones(n, dtype=torch.long)
    cr = make_cond_ablation(yr, mr[:n], c_dim=cfg.get("c_dim", 5)).to(device).float()

    with torch.no_grad():
        x_hat = model.reconstruct(
            Xr[:n].to(device).float(),
            cr,
            m_channel=mr[:n].to(device).float() if cfg.get("use_output_mask_in_decode", True) else None,
            use_mean=True,
        ).detach().cpu()

    t = np.arange(Xr.size(-1)) / FS

    for k in range(n):
        fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True)

        for ci, ch in enumerate(CHANNELS):
            ax = axes[ci]
            if mr[k, ci] > 0.5:
                ax.plot(t, Xr[k, ci].numpy(), label="Real", linewidth=1.2)
                ax.plot(t, x_hat[k, ci].numpy(), label="Recon", linewidth=1.0, alpha=0.8)
            ax.set_title(ch, loc="left", fontsize=9)
            ax.grid(alpha=0.2)

        axes[0].legend(loc="upper right")
        axes[-1].set_xlabel("Time (sec)")
        fig.suptitle(f"Real vs Reconstruction: {meta['name']} | {split_name} sample {k}")
        fig.tight_layout()

        path = os.path.join(out_dir, f"{meta['name']}_recon_{k}.png")
        fig.savefig(path, dpi=160)
        plt.close(fig)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out_dir


# ------------------------------------------------------------
# 1~3. Generator training metric + latent diagnostic + sample stats
# ------------------------------------------------------------
meta_list = load_all_ablation_meta(ABL_ROOT)

summary_rows = []
for meta in meta_list:
    row = {
        "variant": meta["name"],
        "question": meta.get("question", ""),
        "best_epoch": meta.get("best_epoch", np.nan),
        "best_val": meta.get("best_val", np.nan),
        "trainable_params": meta.get("trainable_params", np.nan),
        "ckpt_path": meta.get("ckpt_path", ""),
        "synthetic_path": meta.get("synthetic_path", ""),
    }

    # Training component losses from checkpoint history
    row.update(extract_best_val_components(meta["ckpt_path"]))

    # Latent diagnostics
    latent = meta.get("latent_stats", {})
    row.update({
        "mu_mean": latent.get("mu_mean", np.nan),
        "mu_std": latent.get("mu_std", np.nan),
        "posterior_std_mean": latent.get("std_mean", np.nan),
        "posterior_std_std": latent.get("std_std", np.nan),
        "logvar_mean": latent.get("logvar_mean", np.nan),
    })

    # Generated sample stats
    sample = meta.get("sample_stats", {})
    row.update({
        "n_syn": sample.get("n_syn", np.nan),
        "global_mean": sample.get("global_mean", np.nan),
        "global_std": sample.get("global_std", np.nan),
        "global_min": sample.get("global_min", np.nan),
        "global_max": sample.get("global_max", np.nan),
        "ECG1_std": sample.get("ECG1_std", np.nan),
        "ECG2_std": sample.get("ECG2_std", np.nan),
        "PPG_std": sample.get("PPG_std", np.nan),
        "ABP_std": sample.get("ABP_std", np.nan),
        "ECG1_avail_n": sample.get("ECG1_avail_n", np.nan),
        "ECG2_avail_n": sample.get("ECG2_avail_n", np.nan),
        "PPG_avail_n": sample.get("PPG_avail_n", np.nan),
        "ABP_avail_n": sample.get("ABP_avail_n", np.nan),
    })

    summary_rows.append(row)

ablation_summary = pd.DataFrame(summary_rows)
ablation_summary = ablation_summary.sort_values("variant").reset_index(drop=True)

summary_path = os.path.join(ANALYSIS_ROOT, "ablation_generator_latent_sample_summary.csv")
ablation_summary.to_csv(summary_path, index=False)

print("Saved:", summary_path)
display(ablation_summary)


# ------------------------------------------------------------
# 4. Realism metrics
# ------------------------------------------------------------
real_X, real_m, real_split_name = get_real_true_reference()
print("Real reference:", real_split_name, real_X.shape)

real_feat = extract_feature_matrix(real_X, real_m, max_n=MMD_N)
real_feat_scaled_base = real_feat.copy()

realism_rows = []

for meta in meta_list:
    name = meta["name"]
    syn_path = meta["synthetic_path"]

    if not os.path.exists(syn_path):
        print("[skip] synthetic missing:", name, syn_path)
        continue

    syn_X, syn_m, payload = load_synthetic_payload(syn_path)

    # PSD
    psd_stats = compute_psd_distance(real_X, real_m, syn_X, syn_m)

    # MMD on standardized feature space
    syn_feat = extract_feature_matrix(syn_X, syn_m, max_n=MMD_N)

    scaler = StandardScaler()
    both = np.vstack([real_feat_scaled_base, syn_feat])
    both = scaler.fit_transform(both)

    real_scaled = both[: real_feat_scaled_base.shape[0]]
    syn_scaled = both[real_feat_scaled_base.shape[0] :]

    mmd2 = rbf_mmd2(real_scaled, syn_scaled, max_n=MMD_N)

    # ECG-pulse lag consistency
    real_lag = compute_lag_consistency(real_X, real_m, fs=FS, max_n=MAX_REALISM_N)
    syn_lag = compute_lag_consistency(syn_X, syn_m, fs=FS, max_n=MAX_REALISM_N)

    row = {
        "variant": name,
        "real_reference": real_split_name,
        "mmd2_feature": mmd2,
        **psd_stats,
    }

    for k, v in syn_lag.items():
        row[f"syn_{k}"] = v
    for k, v in real_lag.items():
        row[f"real_{k}"] = v
        if k in syn_lag:
            row[f"absdiff_{k}"] = abs(syn_lag[k] - v) if np.isfinite(syn_lag[k]) and np.isfinite(v) else np.nan

    realism_rows.append(row)

    # Plots
    plot_waveform_overlay(real_X, real_m, syn_X, syn_m, name, n=PLOT_N)

    # Optional reconstruction overlay
    try:
        plot_reconstruction_overlay_for_variant(meta, n=PLOT_N)
    except Exception as e:
        print(f"[warn] reconstruction overlay failed for {name}: {repr(e)}")

realism_df = pd.DataFrame(realism_rows)
realism_df = realism_df.sort_values("variant").reset_index(drop=True)

realism_path = os.path.join(ANALYSIS_ROOT, "ablation_realism_metrics.csv")
realism_df.to_csv(realism_path, index=False)

print("Saved:", realism_path)
display(realism_df)


# ------------------------------------------------------------
# PCA / UMAP-style feature visualization
# PCA is always available. UMAP is optional.
# ------------------------------------------------------------
pca_out_dir = os.path.join(PLOT_ROOT, "pca_umap")
os.makedirs(pca_out_dir, exist_ok=True)

pca_rows = []
labels = []

# real
rf = extract_feature_matrix(real_X, real_m, max_n=250)
pca_rows.append(rf)
labels.extend(["Real"] * rf.shape[0])

# generated variants
for meta in meta_list:
    syn_path = meta["synthetic_path"]
    if not os.path.exists(syn_path):
        continue

    syn_X, syn_m, _ = load_synthetic_payload(syn_path)
    sf = extract_feature_matrix(syn_X, syn_m, max_n=250)
    pca_rows.append(sf)
    labels.extend([meta["name"]] * sf.shape[0])

Fmat = np.vstack(pca_rows)
labels = np.asarray(labels)

Fscaled = StandardScaler().fit_transform(Fmat)
coords = PCA(n_components=2, random_state=42).fit_transform(Fscaled)

pca_df = pd.DataFrame({
    "pc1": coords[:, 0],
    "pc2": coords[:, 1],
    "label": labels,
})
pca_csv = os.path.join(ANALYSIS_ROOT, "ablation_pca_features.csv")
pca_df.to_csv(pca_csv, index=False)

plt.figure(figsize=(9, 7))
for lab in sorted(set(labels)):
    idx = labels == lab
    plt.scatter(coords[idx, 0], coords[idx, 1], s=10, alpha=0.55, label=lab)

plt.title("PCA of Real vs Generated Ablation Features")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(fontsize=7, markerscale=2, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

pca_png = os.path.join(pca_out_dir, "ablation_pca_real_vs_generated.png")
plt.savefig(pca_png, dpi=180)
plt.show()

print("Saved:", pca_csv)
print("Saved:", pca_png)


# Optional UMAP if installed
try:
    import umap

    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=20, min_dist=0.1)
    umap_coords = reducer.fit_transform(Fscaled)

    umap_df = pd.DataFrame({
        "umap1": umap_coords[:, 0],
        "umap2": umap_coords[:, 1],
        "label": labels,
    })
    umap_csv = os.path.join(ANALYSIS_ROOT, "ablation_umap_features.csv")
    umap_df.to_csv(umap_csv, index=False)

    plt.figure(figsize=(9, 7))
    for lab in sorted(set(labels)):
        idx = labels == lab
        plt.scatter(umap_coords[idx, 0], umap_coords[idx, 1], s=10, alpha=0.55, label=lab)

    plt.title("UMAP of Real vs Generated Ablation Features")
    plt.xlabel("UMAP1")
    plt.ylabel("UMAP2")
    plt.legend(fontsize=7, markerscale=2, bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()

    umap_png = os.path.join(pca_out_dir, "ablation_umap_real_vs_generated.png")
    plt.savefig(umap_png, dpi=180)
    plt.show()

    print("Saved:", umap_csv)
    print("Saved:", umap_png)

except Exception as e:
    print("[UMAP skipped]", repr(e))


# ------------------------------------------------------------
# 5. Utility eval manifest
# ------------------------------------------------------------
# This does not train CNN1D/FCN here.
# Use this CSV as input for the downstream utility evaluation notebook.
utility_rows = []

for meta in meta_list:
    if not os.path.exists(meta["synthetic_path"]):
        continue

    utility_rows.append({
        "method": meta["name"],
        "question": meta.get("question", ""),
        "synthetic_path": meta["synthetic_path"],
        "ckpt_path": meta["ckpt_path"],
        "best_val": meta.get("best_val", np.nan),
    })

utility_manifest_df = pd.DataFrame(utility_rows).sort_values("method")
utility_manifest_path = os.path.join(ANALYSIS_ROOT, "ablation_utility_eval_manifest.csv")
utility_manifest_df.to_csv(utility_manifest_path, index=False)

print("Saved utility eval manifest:", utility_manifest_path)
display(utility_manifest_df)


# ------------------------------------------------------------
# Compact interpretation helper table
# ------------------------------------------------------------
compact_cols = [
    "variant",
    "best_val",
    "val_recon",
    "val_wav",
    "val_deriv",
    "val_phys",
    "val_kl",
    "mu_std",
    "posterior_std_mean",
    "global_std",
    "ECG1_std",
    "ECG2_std",
    "PPG_std",
    "ABP_std",
]

compact_df = ablation_summary[[c for c in compact_cols if c in ablation_summary.columns]].copy()

# Add realism compact columns
if len(realism_df) > 0:
    add_cols = ["variant", "mmd2_feature", "psd_mean",
                "syn_PPG_peak_corr_median", "syn_PPG_lag_median_sec", "syn_PPG_phys_lag_rate",
                "syn_ABP_peak_corr_median", "syn_ABP_lag_median_sec", "syn_ABP_phys_lag_rate"]
    realism_compact = realism_df[[c for c in add_cols if c in realism_df.columns]].copy()
    compact_df = compact_df.merge(realism_compact, on="variant", how="left")

compact_path = os.path.join(ANALYSIS_ROOT, "ablation_compact_report_table.csv")
compact_df.to_csv(compact_path, index=False)

print("Saved compact report table:", compact_path)
display(compact_df)

ABL_ROOT: /content/drive/MyDrive/vtac_project/outputs/cvae_ablation
ANALYSIS_ROOT: /content/drive/MyDrive/vtac_project/outputs/cvae_ablation/posthoc_analysis
device: cuda
Saved: /content/drive/MyDrive/vtac_project/outputs/cvae_ablation/posthoc_analysis/ablation_generator_latent_sample_summary.csv


,variant,question,best_epoch,best_val,trainable_params,ckpt_path,synthetic_path,val_loss,val_recon,val_deriv,...,global_min,global_max,ECG1_std,ECG2_std,PPG_std,ABP_std,ECG1_avail_n,ECG2_avail_n,PPG_avail_n,ABP_avail_n
0,arch_concat_fusion,Concatenation fusion vs Attention fusion,4,0.816465,73441188,/content/drive/MyDrive/vtac_project/outputs/cv...,/content/drive/MyDrive/vtac_project/outputs/cv...,0.816465,0.753537,0.024445,...,-3.333955,3.344923,0.399826,0.414479,0.612250,0.603842,1607,1607,1405,605
1,arch_single_encoder,Single encoder vs Multi-branch encoder,7,0.652329,62137124,/content/drive/MyDrive/vtac_project/outputs/cv...,/content/drive/MyDrive/vtac_project/outputs/cv...,0.652329,0.601967,0.024988,...,-4.831288,4.490359,0.558283,0.607659,0.762869,0.638869,1607,1607,1405,605
2,full_proposed,Reference: multi-branch + attention + all loss...,5,0.844836,74229924,/content/drive/MyDrive/vtac_project/outputs/cv...,/content/drive/MyDrive/vtac_project/outputs/cv...,0.844836,0.780071,0.024809,...,-3.603169,3.661839,0.399416,0.429522,0.592165,0.621812,1607,1607,1405,605
3,loss_no_coeff_wavelet,w/o coefficient-domain loss,5,0.821604,74229924,/content/drive/MyDrive/vtac_project/outputs/cv...,/content/drive/MyDrive/vtac_project/outputs/cv...,0.821604,0.777594,0.024837,...,-3.981812,4.310860,0.412778,0.435301,0.601793,0.597593,1607,1607,1405,605
4,loss_no_derivative,w/o derivative loss,5,0.827419,74229924,/content/drive/MyDrive/vtac_project/outputs/cv...,/content/drive/MyDrive/vtac_project/outputs/cv...,0.827419,0.776215,0.024773,...,-3.630769,3.976651,0.408333,0.433407,0.603305,0.616334,1607,1607,1405,605
5,loss_no_phys_feature,w/o ECG-pulse / classifier feature consistency...,9,0.836579,74229924,/content/drive/MyDrive/vtac_project/outputs/cv...,/content/drive/MyDrive/vtac_project/outputs/cv...,0.836579,0.783557,0.026038,...,-4.182596,3.909928,0.488076,0.524273,0.780768,0.694324,1607,1607,1405,605
6,mask_no_channel_mask_condition,w/o channel mask condition,5,0.885465,74223396,/content/drive/MyDrive/vtac_project/outputs/cv...,/content/drive/MyDrive/vtac_project/outputs/cv...,0.885465,0.816647,0.024648,...,-3.978864,3.956876,0.400558,0.438044,0.563909,0.502433,1607,1607,1405,605


Real reference: test_true torch.Size([137, 4, 2500])
